# Digital Banking Fraud Detection — Full Pipeline
**Team:** Group A (Data) -> Group B (ML) -> Group C (Evaluation & Deployment)
**Dataset:** Bank Transaction Fraud Detection (Kaggle, marusagar) — `bank_transaction_fraud_detection.csv`

> This notebook is **column-agnostic by design**. I don't have your actual CSV to verify exact
> column names against, so instead of hardcoding names that might be slightly wrong, the cleaning/
> feature-engineering steps auto-detect: the target column, ID-like columns to drop, date/time
> columns to convert, and which remaining columns are numeric vs categorical. Run Section 2 first
> and **read the printed output** — confirm the detected target column and dropped columns look
> right before going further. If something's off, there's a manual override at the top of Section 4.

## How to get the dataset
1. https://www.kaggle.com/datasets/marusagar/bank-transaction-fraud-detection -> download the CSV
2. Rename it to `bank_transaction_fraud_detection.csv` (or just change DATA_PATH below) and place
   it next to this notebook / upload it in Colab


## 1. Setup

In [ ]:
# If running in Colab, uncomment:
# !pip install imbalanced-learn xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve)
from imblearn.over_sampling import SMOTE
import joblib

sns.set_style("whitegrid")
RANDOM_STATE = 42


## 2. Data Collection & Understanding (Group A — Member 1)
**Deliverables:** dataset loaded, feature descriptions, problem statement

**Problem statement:** Given a bank transaction (amount, account balance, customer demographics,
transaction type/channel/location, device info, etc.), predict whether it's fraudulent. Like most
fraud datasets, expect heavy class imbalance — confirm the actual ratio below before assuming
accuracy is a usable metric.


In [ ]:
DATA_PATH = "bank_transaction_fraud_detection.csv"  # adjust if your filename differs

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("\nColumns:\n", list(df.columns))
df.head()


In [ ]:
# Auto-detect the target column — override TARGET_COL manually if this guesses wrong
CANDIDATE_TARGET_NAMES = ['is_fraud', 'isfraud', 'fraud', 'class', 'target', 'label']
TARGET_COL = next((c for c in df.columns if c.lower().replace(' ', '_') in CANDIDATE_TARGET_NAMES), None)

if TARGET_COL is None:
    raise ValueError(
        "Couldn't auto-detect the target column. Look at the column list above and set "
        "TARGET_COL manually, e.g. TARGET_COL = 'Is_Fraud'"
    )
print(f"Detected target column: '{TARGET_COL}'  <-- confirm this is right")
df[TARGET_COL].value_counts()


In [ ]:
df.info()
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
print("\nFraud %:", round(df[TARGET_COL].mean() * 100, 4), "%")


## 3. Data Cleaning & EDA (Group A — Member 2)
**Deliverables:** charts/visualizations, insights report


In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows")

# Drop rows missing the target — can't train/evaluate on those
df = df.dropna(subset=[TARGET_COL])


In [ ]:
# Class imbalance
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=TARGET_COL, data=df, ax=ax[0])
ax[0].set_title('Class Distribution (raw counts)')
df[TARGET_COL].value_counts(normalize=True).plot(kind='bar', ax=ax[1])
ax[1].set_title('Class Distribution (%)')
plt.tight_layout()
plt.show()


In [ ]:
# Numeric feature distributions, fraud vs legit
numeric_preview = df.select_dtypes(include=[np.number]).columns.drop(TARGET_COL, errors='ignore')
n = min(len(numeric_preview), 6)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]
for ax, col in zip(axes, numeric_preview[:n]):
    sns.boxplot(x=TARGET_COL, y=col, data=df, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
# Fraud rate by categorical feature (top few categorical columns)
categorical_preview = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns found:", categorical_preview)

for col in categorical_preview[:4]:
    if df[col].nunique() <= 15:  # skip high-cardinality columns (likely IDs/names) for this chart
        rates = df.groupby(col)[TARGET_COL].mean().sort_values(ascending=False)
        plt.figure(figsize=(8, 3))
        rates.plot(kind='bar', color='crimson')
        plt.title(f'Fraud rate by {col}')
        plt.ylabel('Fraud rate')
        plt.tight_layout()
        plt.show()


In [ ]:
# Correlation heatmap (numeric features only)
num_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(8, 6))
sns.heatmap(num_df.corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Numeric Feature Correlation')
plt.tight_layout()
plt.show()


**Insights (fill in after running, based on your actual charts):**
- Note the fraud rate (%) — this tells you how aggressively you need to handle imbalance
- Call out which categorical groups (transaction type, channel, account type, etc.) show
  disproportionately higher fraud rates — that's a strong finding for the report
- Note any numeric feature (amount, balance, age, duration) where fraud vs legit distributions
  visibly diverge


## 4. Feature Engineering & Data Preparation (Group B — Member 1)
**Deliverables:** processed dataset, feature engineering notebook


In [ ]:
# --- Manual overrides (only touch these if auto-detection looks wrong in Section 2/3) ---
MANUAL_DROP_COLS = []   # e.g. ['Customer_ID'] if it slips through the heuristic below
# ---------------------------------------------------------------------------------------

# Heuristic: drop ID-like / free-text columns that shouldn't be model features
ID_LIKE_SUBSTRINGS = ['id', 'name', 'email', 'contact', 'description', 'address', 'phone']
id_like_cols = [
    c for c in df.columns
    if c != TARGET_COL and any(s in c.lower() for s in ID_LIKE_SUBSTRINGS)
]

# Heuristic: date/time columns -> engineer simple features, then drop the raw column
datetime_cols = [
    c for c in df.columns
    if c != TARGET_COL and c not in id_like_cols and ('date' in c.lower() or 'time' in c.lower())
]

print("Dropping as ID-like:", id_like_cols)
print("Treating as date/time (will engineer features from these):", datetime_cols)

work = df.drop(columns=id_like_cols + MANUAL_DROP_COLS, errors='ignore')

for col in datetime_cols:
    parsed = pd.to_datetime(work[col], errors='coerce')
    if parsed.notna().mean() > 0.5:  # only keep if most values parsed successfully
        work[f'{col}_hour'] = parsed.dt.hour
        work[f'{col}_dayofweek'] = parsed.dt.dayofweek
    work = work.drop(columns=[col])

print("\nFinal columns going into modeling:\n", list(work.columns))


In [ ]:
# Split numeric vs categorical (everything except target)
numeric_cols = [c for c in work.select_dtypes(include=[np.number]).columns if c != TARGET_COL]
categorical_cols = [c for c in work.select_dtypes(include=['object']).columns if c != TARGET_COL]

print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)

work[numeric_cols] = work[numeric_cols].fillna(work[numeric_cols].median())
for c in categorical_cols:
    work[c] = work[c].fillna('Unknown')

# Save the exact categories seen at training time -> needed so app.py can build matching dropdowns
categorical_options = {c: sorted(work[c].astype(str).unique().tolist()) for c in categorical_cols}
joblib.dump(categorical_options, "categorical_options.pkl")
joblib.dump(numeric_cols, "numeric_columns.pkl")


In [ ]:
scaler = StandardScaler()
work[numeric_cols] = scaler.fit_transform(work[numeric_cols])

X = pd.get_dummies(work.drop(columns=[TARGET_COL]), columns=categorical_cols)
y = work[TARGET_COL].astype(int)

feature_columns = list(X.columns)
joblib.dump(feature_columns, "feature_columns.pkl")
joblib.dump(scaler, "scaler.pkl")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train shape:", X_train.shape, "Fraud %:", round(y_train.mean()*100, 4))
print("Test shape:", X_test.shape, "Fraud %:", round(y_test.mean()*100, 4))


In [ ]:
# Handle class imbalance with SMOTE — ONLY on training data
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_res.value_counts().to_dict())


## 5. Machine Learning Model Development (Group B — Member 2)
**Deliverables:** trained models, accuracy results


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE),
}

trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_res, y_train_res)
    trained_models[name] = model
print("Done.")


## 6. Model Evaluation & Interpretation (Group C — Member 1, **your task**)
**Deliverables:** performance comparison table, evaluation report

Evaluated on the **original, untouched** test set (no SMOTE) — the only way to know how the
model performs under real-world class imbalance.


In [ ]:
results = []
roc_data = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[name] = (fpr, tpr, results[-1]["ROC-AUC"])

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
results_df


In [ ]:
fig, axes = plt.subplots(1, len(trained_models), figsize=(5 * len(trained_models), 4))
for ax, (name, model) in zip(axes, trained_models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 6))
for name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()


**Why ROC-AUC and Recall matter most here:** missing a real fraud (false negative) is usually
far more costly than flagging a legit transaction for review (false positive). Pick the "best"
model by Recall/ROC-AUC, not raw accuracy, and say so explicitly in your report.


In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
print(f"Best model by ROC-AUC: {best_model_name}")
print(classification_report(y_test, best_model.predict(X_test), target_names=['Legit', 'Fraud']))


## 7. Export for Deployment (feeds into Group C — Member 2 / `app.py`)

In [ ]:
joblib.dump(best_model, "fraud_model.pkl")
print(f"Saved {best_model_name} -> fraud_model.pkl")
print("Also saved: scaler.pkl, feature_columns.pkl, numeric_columns.pkl, categorical_options.pkl")
